# 🖍️ ShinChan Life Simulator - GRPO Training with OpenEnv
Train a small language model to make decisions as Shin-chan using Reinforcement Learning!

In [ ]:
# Bootstrap repo + dependencies (robust Colab setup)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Sarthaks-24/sinchan_env.git"
REPO_DIR = Path("/content/sinchan_env")

# 1) Core dependencies
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "openenv-core[core]>=0.2.2",
    "trl",
    "datasets",
    "torch",
    "transformers",
    "matplotlib",
])

# 2) Ensure repository exists in /content
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

# 3) Validate project root and editable install
if not (REPO_DIR / "pyproject.toml").exists():
    raise FileNotFoundError(f"pyproject.toml not found at {REPO_DIR}. Check REPO_URL.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
os.chdir(REPO_DIR)

print("Repository:", REPO_DIR)
print("Working dir:", os.getcwd())
print("Setup complete.")

### Step 1: Connect to the Environment
Replace the URL below with your Hugging Face Space URL.

Tip: use the `https://<username>-<space-name>.hf.space` runtime URL (not the repo URL).

This notebook now includes an automatic fallback: if HF WebSocket is unstable, it starts the environment locally in Colab and continues training on `http://127.0.0.1:8000`.

In [ ]:
import os
# Set to your deployed Hugging Face Space URL
os.environ["ENV_URL"] = "https://gladiator-codes-sinchan-env.hf.space"
print("ENV_URL:", os.environ["ENV_URL"])

In [ ]:
# Connectivity probe with permanent fallback:
# 1) Try HF Space (using HTTP GET first to wake it up)
# 2) If unstable, auto-start local server in Colab and continue on localhost
import os
import time
import subprocess
import sys
from pathlib import Path

import requests
from sinchan_env import SinChanEnv


def probe_env(base_url: str, attempts: int = 8):
    """Try reset + get_scenario_info over OpenEnv client with retries."""
    
    # 1. Wake up the Hugging Face space via HTTP GET if it's sleeping
    health_url = f"{base_url.rstrip('/')}/health"
    print(f"Waking up and checking health at {health_url}...")
    for idx in range(1, attempts + 1):
        try:
            r = requests.get(health_url, timeout=5)
            if r.status_code == 200:
                print("HTTP Health check passed! Space is awake.")
                break
            print(f"[Health check {idx}/{attempts}] Status {r.status_code}. Waiting 10s...")
        except Exception as e:
            print(f"[Health check {idx}/{attempts}] Error: {e}. Waiting 10s...")
        time.sleep(10)

    # 2. Proceed with WebSocket probe
    last_err = None
    for idx in range(1, attempts + 1):
        try:
            client = SinChanEnv(base_url=base_url.rstrip("/"))
            sync_ctx = client.sync() if hasattr(client, "sync") else client
            with sync_ctx as env:
                env.reset()
                info = env.call_tool("get_scenario_info")
                title = info.get("title", "unknown") if isinstance(info, dict) else "unknown"
                print(f"Probe OK at {base_url}. Scenario: {title}")
            return True, None
        except Exception as exc:
            last_err = exc
            wait = min(30, 5 * idx)
            print(f"[Probe retry {idx}/{attempts}] {base_url} not ready, waiting {wait}s... Error: {exc}")
            time.sleep(wait)
    return False, last_err


def start_local_server() -> subprocess.Popen:
    """Start uvicorn server in background inside Colab."""
    repo_dir = Path("/content/sinchan_env")
    
    # Disable Web UI for local Colab server to avoid missing 'gradio' dependency crashes
    env_vars = os.environ.copy()
    env_vars["ENABLE_WEB_INTERFACE"] = "false"
    
    cmd = [
        sys.executable,
        "-m",
        "uvicorn",
        "server.app:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ]
    print("Starting local fallback server:", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=str(repo_dir),
        env=env_vars,
        stdout=sys.stdout,  # Print stdout to notebook so we can see what happens
        stderr=sys.stderr,  # Print stderr to notebook
    )

    # Wait for local /health to come up
    local_health = "http://127.0.0.1:8000/health"
    for _ in range(30):
        try:
            r = requests.get(local_health, timeout=2)
            if r.status_code == 200:
                print("Local server is healthy at", local_health)
                return proc
        except Exception:
            pass
        time.sleep(1)

    proc.terminate()
    raise RuntimeError("Local fallback server failed to start on :8000")


remote_url = os.environ["ENV_URL"].rstrip("/")
print("Primary ENV_URL:", remote_url)

ok, err = probe_env(remote_url, attempts=8)

if not ok:
    print("Remote Space unstable. Switching to local fallback for training.")
    _local_server_proc = start_local_server()
    os.environ["ENV_URL"] = "http://127.0.0.1:8000"
    print("ENV_URL switched to", os.environ["ENV_URL"])
    ok2, err2 = probe_env(os.environ["ENV_URL"], attempts=20)
    if not ok2:
        raise RuntimeError(f"Both remote and local probes failed. Remote error: {err}; Local error: {err2}")


### Step 2: Run Training (Configurable)

In [ ]:
# Run training with explicit args for reproducibility
import os
import subprocess
import sys

TRAIN_SCRIPT = "/content/sinchan_env/training/train_sinchan.py"
OUTPUT_DIR = "/content/sinchan_env/training/artifacts/run1"

if not os.path.exists(TRAIN_SCRIPT):
    raise FileNotFoundError(f"Training script not found: {TRAIN_SCRIPT}")

cmd = [
    sys.executable,
    TRAIN_SCRIPT,
    "--env-url", os.environ["ENV_URL"],
    "--max-steps", "200",
    "--dataset-size", "200",
    "--learning-rate", "1e-5",
    "--num-generations", "2",
    "--output-dir", OUTPUT_DIR,
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

### Step 3: Generate Evaluation Summary
Run a quick baseline comparison and save JSON evidence.

In [ ]:
import os
import subprocess
import sys

EVAL_SCRIPT = "/content/sinchan_env/training/evaluate_scenarios.py"
EVAL_OUTPUT = "/content/sinchan_env/training/artifacts/eval_summary.json"

if not os.path.exists(EVAL_SCRIPT):
    raise FileNotFoundError(f"Eval script not found: {EVAL_SCRIPT}")

cmd = [
    sys.executable,
    EVAL_SCRIPT,
    "--env-url", os.environ["ENV_URL"],
    "--episodes", "10",
    "--output", EVAL_OUTPUT,
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

### Step 4: Generate Plot PNGs for Submission
Creates reward/loss/baseline comparison charts in `assets/`.

In [ ]:
import os
import subprocess
import sys

PLOT_SCRIPT = "/content/sinchan_env/training/plot_metrics.py"
RUN_DIR = "/content/sinchan_env/training/artifacts/run1"
EVAL_SUMMARY = "/content/sinchan_env/training/artifacts/eval_summary.json"
ASSETS_DIR = "/content/sinchan_env/assets"

if not os.path.exists(PLOT_SCRIPT):
    raise FileNotFoundError(f"Plot script not found: {PLOT_SCRIPT}")

cmd = [
    sys.executable,
    PLOT_SCRIPT,
    "--run-dir", RUN_DIR,
    "--eval-summary", EVAL_SUMMARY,
    "--assets-dir", ASSETS_DIR,
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)